In [15]:
import pandas as pd
import numpy as np
import re

# -----------------------------------
# 1. LOAD + CLEAN ORIGINAL DATA
# -----------------------------------
df = pd.read_csv("combined_calls.csv", low_memory=False)
df["Date"] = pd.to_datetime(df["Activity Start Timestamp"], errors="coerce")
df["Year"]  = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"]   = df["Date"].dt.date

df["Contact Session ID"] = df["Contact Session ID"].astype(str).str.strip()

# -----------------------------------
# 2. MENU CLASSIFICATION (per row)
# -----------------------------------
menu_patterns = {
    "MainMenu": r"Main",
    "SuburbanSeniors": r"Suburban",
    "CitySeniors": r"City",
    "HIVMenu": r"HIVMenu",
    "Family": r"Family",
    "Housing": r"Housing",
    "Benefits": r"Benefits",
    "Consumer": r"Consumer",
    "Criminal Records": r"Criminal Records",
    "Employment": r"Employment",
    "Immigration": r"Immigration",
    "ADAPT": r"ADAPT"
}

def classify_menu(activity_name): 
    if pd.isna(activity_name):
        return None
    name = str(activity_name)
    for menu, pattern in menu_patterns.items():
        if re.search(pattern, name, re.IGNORECASE):
            return menu
    return None

df["Menu_Category"] = df["Activity Name"].apply(classify_menu)

# -----------------------------------
# 3. FIND LAST MENU CATEGORY PER SESSION
# -----------------------------------
df_sorted = df.sort_values(["Contact Session ID", "Date"])

last_menu = (
    df_sorted[df_sorted["Menu_Category"].notna()]
    .groupby("Contact Session ID", as_index=False)
    .last()[["Contact Session ID", "Menu_Category"]]
)

# -----------------------------------
# 4. GET YEAR AND MONTH PER SESSION
# -----------------------------------
last_date = (
    df_sorted
    .groupby("Contact Session ID", as_index=False)
    .last()[["Contact Session ID", "Year", "Month"]]
)

# -----------------------------------
# 5. SUBTYPE CLASSIFICATION FROM ORIGINAL DATA
# -----------------------------------
def determine_subtype(group):
    """
    Priority logic:
    1. Agent Answered - if ANY row has an agent (takes priority)
    2. Voicemail - if LAST non-missing Activity Name == 'ClinicVoicemailTransfer' AND no agent
    3. Closed Queue - if any activity contains closedqueue/closed queue
    4. Other - everything else
    """
    has_agent = group["Agent Name"].notna().any() and (group["Agent Name"].str.strip() != "").any()
    if has_agent:
        return "Agent Answered"
    
    non_missing_activities = group[group["Activity Name"].notna()]
    if len(non_missing_activities) > 0:
        last_activity = non_missing_activities.sort_values("Activity Start Timestamp").iloc[-1]["Activity Name"]
        
        if str(last_activity).lower() == "clinicvoicemailtransfer":
            return "Voicemail"
    
    all_activities = group["Activity Name"].dropna().astype(str).str.lower()
    if all_activities.str.contains("closedqueue|closed queue", case=False, na=False).any():
        return "Closed Queue"
    
    return "Other"

session_subtypes = (
    df.groupby("Contact Session ID")
    .apply(determine_subtype)
    .reset_index()
    .rename(columns={0: "Subtype"})
)

# -----------------------------------
# 6. CREATE FIRST DATASET (from transformations)
# -----------------------------------
all_sessions = pd.DataFrame({"Contact Session ID": df["Contact Session ID"].unique()})

dataset1 = all_sessions.merge(last_menu, on="Contact Session ID", how="left")
dataset1 = dataset1.merge(session_subtypes, on="Contact Session ID", how="left")
dataset1 = dataset1.merge(last_date, on="Contact Session ID", how="left")

dataset1["Menu_Category"] = dataset1["Menu_Category"].fillna("Other")
dataset1["Subtype"] = dataset1["Subtype"].fillna("Other")

print("Dataset 1 (from transformations):")
print(f"Shape: {dataset1.shape}")
print(f"Columns: {dataset1.columns.tolist()}")

# -----------------------------------
# 7. LOAD AND PREPARE SECOND DATASET (call_tag)
# -----------------------------------
call_tag = pd.read_csv("call_tag.csv")
call_tag["Contact Session ID"] = call_tag["Contact Session ID"].astype(str).str.strip()

print(f"\nDataset 2 (call_tag):")
print(f"Shape: {call_tag.shape}")
print(f"Columns: {call_tag.columns.tolist()}")

# -----------------------------------
# 8. RECLASSIFY outcome_type IN call_tag
# -----------------------------------
def reclassify_outcome_type(outcome):
    """
    Reclassify outcome_type into 5 categories:
    1. Closed Queue - "Closed queue" or "Closed hours"
    2. Voicemail - any outcome with "voicemail" in it
    3. Agent Answered - "Connected to agent" or "Staff directory"
    4. Abandoned - "Abandoned", "No language selected (FD1)", "No selection on Main menu (FD3)"
    5. Other - everything else
    """
    if pd.isna(outcome):
        return "Other"
    
    outcome_str = str(outcome).lower()
    
    # Check for Closed Queue
    if "closed queue" in outcome_str or "closed hours" in outcome_str:
        return "Closed Queue"
    
    # Check for Voicemail (any type)
    if "voicemail" in outcome_str:
        return "Voicemail"
    
    # Check for Agent Answered
    if "connected to agent" in outcome_str or "staff directory" in outcome_str:
        return "Agent Answered"
    
    # Check for Abandoned
    if outcome_str in ["abandoned"]:
        return "Abandoned"
    
    # Everything else
    return "Other"

call_tag["outcome_type_reclassified"] = call_tag["outcome_type"].apply(reclassify_outcome_type)

print("\n" + "="*70)
print("RECLASSIFIED outcome_type SUMMARY")
print("="*70)
print(call_tag["outcome_type_reclassified"].value_counts())

# -----------------------------------
# 9. MERGE THE TWO DATASETS
# -----------------------------------
# Rename columns in call_tag to avoid conflicts
call_tag_renamed = call_tag.rename(columns={
    "Overall_tag": "Overall_tag_calltag",
    "outcome_type": "outcome_type_original",
    "outcome_type_reclassified": "outcome_type"
})

# Merge on Contact Session ID
merged_dataset = dataset1.merge(
    call_tag_renamed, 
    on="Contact Session ID", 
    how="left"
)

print("\n" + "="*70)
print("MERGED DATASET SUMMARY")
print("="*70)
print(f"Shape: {merged_dataset.shape}")
print(f"Columns: {merged_dataset.columns.tolist()}")

# -----------------------------------
# 10. SUMMARY STATISTICS
# -----------------------------------
print("\n" + "="*70)
print("CROSS-TABULATION: Menu_Category vs outcome_type (from call_tag)")
print("="*70)

crosstab = pd.crosstab(
    merged_dataset['Menu_Category'], 
    merged_dataset['outcome_type'], 
    margins=True,
    margins_name='Total'
)
print(crosstab)

print("\n" + "="*70)
print("COMPARISON: Subtype (from transformations) vs outcome_type (from call_tag)")
print("="*70)

comparison = pd.crosstab(
    merged_dataset['Subtype'], 
    merged_dataset['outcome_type'], 
    margins=True,
    margins_name='Total'
)
print(comparison)

# -----------------------------------
# 11. SAVE MERGED DATASET
# -----------------------------------

# Preview first 10 rows
print("\n" + "="*70)
print("FIRST 10 ROWS OF MERGED DATASET")
print("="*70)
print(merged_dataset.head(10).to_string())

Dataset 1 (from transformations):
Shape: (248442, 5)
Columns: ['Contact Session ID', 'Menu_Category', 'Subtype', 'Year', 'Month']

Dataset 2 (call_tag):
Shape: (248442, 7)
Columns: ['Unnamed: 0', 'Contact Session ID', 'caller_type', 'Overall_tag', 'outcome', 'outcome_type', 'outcome_source']

RECLASSIFIED outcome_type SUMMARY
Abandoned         76739
Closed Queue      68506
Agent Answered    50985
Other             31548
Voicemail         20664
Name: outcome_type_reclassified, dtype: int64

MERGED DATASET SUMMARY
Shape: (248442, 12)
Columns: ['Contact Session ID', 'Menu_Category', 'Subtype', 'Year', 'Month', 'Unnamed: 0', 'caller_type', 'Overall_tag_calltag', 'outcome', 'outcome_type_original', 'outcome_source', 'outcome_type']

CROSS-TABULATION: Menu_Category vs outcome_type (from call_tag)
outcome_type     Abandoned  Agent Answered  Closed Queue  Other  Voicemail  \
Menu_Category                                                                
ADAPT                 2489             570

In [16]:
merged_dataset.head()

,Contact Session ID,Menu_Category,Subtype,Year,Month,Unnamed: 0,caller_type,Overall_tag_calltag,outcome,outcome_type_original,outcome_source,outcome_type
0,001a3748-8d50-4550-8461-33547983deb0,Employment,Closed Queue,2025,1,1228,Non-senior legal issue,Legal issue,Negative,Closed queue,ClosedQueueMenu,Closed Queue
1,00229391-8614-4eb1-b1e0-e0a78076e0e4,Family,Closed Queue,2025,1,1750,Non-senior legal issue,Legal issue,Negative,Closed queue,ClosedQueueMenu,Closed Queue
2,002e5e73-c0c5-42f4-b04e-ee9babc7d88b,MainMenu,Other,2025,1,2548,Non-senior legal issue,Legal issue,Neutral,LAC does not serve,OtherLegalMenu,Other
3,0042f5e6-6c86-4bc1-84a5-4bfee8eb5580,Housing,Closed Queue,2025,1,3613,Chicago senior,Legal issue,Negative,Closed queue,ClosedQueueMenu,Closed Queue
4,0053f689-47d7-43fc-abe9-0ffe71bbdb10,Family,Agent Answered,2025,1,4380,Non-senior legal issue,Legal issue,Negative,Closed queue,ClinicVoicemailTransfer (Closed Queue EP),Closed Queue


In [18]:
# Update Subtype to "Abandoned" where outcome_type is "Abandoned"
merged_dataset.loc[merged_dataset['outcome_type'] == 'Abandoned', 'Subtype'] = 'Abandoned'

# Check the updated distribution
print("Updated Subtype distribution:")
print(merged_dataset['Subtype'].value_counts())

Updated Subtype distribution:
Agent Answered    97272
Abandoned         76739
Closed Queue      42039
Other             32388
Voicemail             4
Name: Subtype, dtype: int64


In [19]:
merged_dataset.to_csv("merged.csv", index=False)